# 05b — OntoBricks GraphRAG MCP Server

This notebook deploys a **graph-backed Retrieval-Augmented Generation (GraphRAG) service** as a Databricks App, then exposes it to the Multi-Agent Supervisor via an MCP (Model Context Protocol) connection. Unlike the Neo4j approach in notebook `05b_create_mcp_neo4j_graph_OPTIONAL`, everything here runs **entirely within the Databricks environment** — no external graph database required.

---

## Architecture

```
┌─────────────────────────────────┐
│  Supervisor Agent (notebook 04) │
│  routes queries to child agents │
└───────────┬─────────────────────┘
            │  MCP protocol (SSE)
            ▼
┌───────────────────────────────────────┐
│  UC Connection: ontobricks-graphrag   │
│  (HTTP + bearer token auth)           │
└───────────┬───────────────────────────┘
            │
            ▼
┌───────────────────────────────────────┐
│  Databricks App (FastMCP server)      │
│  Tools:                               │
│    • query_graph(query)               │
│    • get_company_summary(ticker)      │
│    • compare_companies(tickers)       │
└───────────┬───────────────────────────┘
            │  Spark SQL via
            │  databricks-connect
            ▼
┌───────────────────────────────────────┐
│  Unity Catalog Delta Tables           │
│    • graphrag_vertices (1,757 rows)   │
│    • graphrag_edges    (3,493 rows)   │
└───────────────────────────────────────┘
```

### Why GraphRAG instead of plain SQL?

Graph structures let the Supervisor Agent answer **relationship-centric** questions that are awkward in flat SQL:
* "Which companies traded above their 5-day moving average for the longest streak?" → NEXT_DAY chain traversal
* "Compare the fundamentals of NVDA vs AAPL" → Company node properties
* "Show me TSLA's most volatile trading days" → TradingDay properties via TRADED_ON

The graph schema encodes **temporal chains** (NEXT_DAY) and **ownership links** (TRADED_ON) that make multi-hop reasoning natural.

---

## Graph Schema — Nodes

### Company (7 nodes)
One per Mag-7 stock. Carries the latest fundamental snapshot.

| Property | Type | Example | Description |
|----------|------|---------|-------------|
| `id` / `ticker` | string | `NVDA` | Stock ticker (also the node ID) |
| `market_cap` | long | 3,593,883,705,344 | Market capitalization in USD |
| `beta` | double | 1.637 | Volatility relative to S&P 500 |
| `pe_trailing` | double | 54.52 | Trailing 12-month P/E ratio |
| `pe_forward` | double | 31.25 | Forward P/E ratio |
| `ev_ebitda` | double | 48.27 | Enterprise Value / EBITDA |

### TradingDay (1,750 nodes)
One per (company × trading date). \~250 trading days per company.

| Property | Type | Example | Description |
|----------|------|---------|-------------|
| `id` | string | `NVDA_2024-11-25` | Composite key: `{ticker}_{date}` |
| `ticker` | string | `NVDA` | Owning company |
| `date` | timestamp | 2024-11-25 | Calendar date |
| `price_close` | double | 141.95 | Closing price in USD |
| `volume` | int | 198,766,300 | Shares traded |
| `daily_return` | double | 0.83 | Intraday return % = (close-open)/open×100 |

---

## Graph Schema — Edges

| Relationship | Source → Target | Count | Purpose |
|-------------|-----------------|-------|---------|
| **TRADED_ON** | Company → TradingDay | 1,750 | Links a company to each of its trading days |
| **NEXT_DAY** | TradingDay → TradingDay | 1,743 | Temporal chain within each company (7 fewer = 7 chain starts) |

The **NEXT_DAY** chain enables streak/momentum queries: traverse consecutive days to find winning streaks, drawdowns, or volatility clusters without complex window functions.

---

## MCP Tools Exposed

The Databricks App runs a [FastMCP](https://github.com/jlowin/fastmcp) server over SSE transport with three tools:

| Tool | Signature | Use Case |
|------|-----------|----------|
| `query_graph` | `query_graph(query: str)` | Natural-language graph queries — routes to company lookups, comparisons, or schema info based on keywords |
| `get_company_summary` | `get_company_summary(ticker: str)` | Deep-dive on one company: fundamentals + last 10 trading days |
| `compare_companies` | `compare_companies(tickers: str)` | Side-by-side comparison table; pass comma-separated tickers like `"AAPL,MSFT,NVDA"` |

---

## Cell Execution Order

| Cell | Action | Idempotent? |
|------|--------|-------------|
| 2 | Install `fastmcp` (for local testing only) | Yes |
| 3 | Load config, set names, init SDK client | Yes |
| 4 | Build graph DataFrames from `ticker_data` | Yes |
| 5 | Write `graphrag_vertices` and `graphrag_edges` Delta tables | Yes (overwrite) |
| 6 | Generate MCP app source code string | Yes |
| 7 | Upload files → create app → deploy | Yes (delete-before-upload pattern) |
| 8 | Create UC HTTP connection with bearer token | Skips if exists |
| 9 | Smoke-test the graph tables directly | Yes |

---

## Prerequisites

1. Run **`00_tables_setup`** — creates the `ticker_data` source table in Unity Catalog
2. **Databricks Apps** must be enabled in the workspace (contact admin if `POST /api/2.0/apps` returns 404)
3. **`../config.py`** must define: `catalog`, `schema`, `app_resource_suffix`
4. The executing user needs `CREATE TABLE` on the target schema and `CREATE CONNECTION` on the metastore

## Troubleshooting

| Symptom | Likely Cause | Fix |
|---------|-------------|-----|
| App stays `UNAVAILABLE` for >5 min | Workspace Apps quota or cold start | Check Apps UI; re-run cell 7 |
| `CONNECTION_HTTP_BEARER` error | Missing `bearer_token` in connection options | Cell 8 includes the token automatically |
| MCP tools return empty results | `graphrag_vertices` table missing or empty | Re-run cells 4–5 |
| Supervisor can't reach MCP | Connection name mismatch | Verify `ontobricks-graphrag` in cell 8 output matches notebook 04 config |

In [0]:
# --- Dependencies ---
# fastmcp is only needed for local testing/validation of the MCP server. The Databricks App installs its
# own copy from requirements.txt (see cell 6). The OntoBricks/ontos package is NOT installed here because
# it runs inside the Databricks App runtime, not in this notebook's compute context.
%pip install fastmcp --quiet
dbutils.library.restartPython()

In [0]:
# --- Imports & Configuration ---
# Standard libraries plus PySpark for graph DataFrame construction (cells 4-5) and the Databricks SDK for
# REST API calls to manage Apps and Connections (cells 7-8).
import sys
import os
import json
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from databricks.sdk import WorkspaceClient

# Load shared project config (catalog, schema, app_resource_suffix, sa_name). These variables are defined
# in ../config.py and shared across all setup notebooks.
sys.path.append('..')
exec(open('../config.py').read())

CATALOG = catalog
SCHEMA = schema

# Derive app and connection names from the shared suffix so they're unique per workshop participant but
# deterministic for re-runs and cross-notebook references. Both names include the suffix to prevent
# collisions when multiple users share a workspace. Notebook 04 discovers connections by scanning UC, so
# the suffixed name is picked up automatically.
GRAPHRAG_APP_NAME = f"ontobricks-graphrag-{app_resource_suffix}"
MCP_CONNECTION_NAME = f"ontobricks-graphrag-{app_resource_suffix}"

# WorkspaceClient auto-authenticates using the notebook's execution context. We use w.api_client.do() for
# REST calls throughout because the SDK's high-level methods (w.apps.*, w.connections.*) are incomplete on
# serverless compute.
w = WorkspaceClient()
print(f"Catalog: {CATALOG}")
print(f"Schema:  {SCHEMA}")
print(f"App:     {GRAPHRAG_APP_NAME}")
print(f"MCP:     {MCP_CONNECTION_NAME}")

In [0]:
# --- Build Graph DataFrames ---
# Transform the flat ticker_data table (created by 00_tables_setup) into a property-graph representation
# with two vertex types and two edge types. This mirrors the Neo4j schema from notebook
# 05b_create_mcp_neo4j_graph but stores everything as Delta tables the Databricks App queries via Spark SQL.

# ── Source Data ──
# ticker_data has one row per (company × trading day) with columns: Date, company_name, price_open,
# price_close, volume, pe_trailing, pe_forward, peg, ev_ebitda, market_cap, beta
ticker_df = spark.table(f"{CATALOG}.{SCHEMA}.ticker_data")
print(f"Ticker data: {ticker_df.count()} rows")

# Get the universe of companies (should be 7 Mag-7 stocks)
companies = [row.company_name for row in ticker_df.select("company_name").distinct().collect()]
print(f"Companies: {companies}")

# ── Company Vertices ──
# One node per company with static fundamental metrics. These values are the latest snapshot from Yahoo
# Finance (loaded by 00_tables_setup). We use company_name as the node ID for simple lookups.
companies_df = ticker_df.select(
    F.col("company_name").alias("id"),
    F.lit("Company").alias("type"),
    F.col("company_name").alias("ticker"),
    "market_cap", "beta", "pe_trailing", "pe_forward", "ev_ebitda"
).dropDuplicates(["id"])

# ── TradingDay Vertices ──
# One node per (company × date) with intraday price data. The composite ID format "TICKER_YYYY-MM-DD"
# enables efficient filtering and ensures uniqueness across companies. daily_return = intraday %.
trading_days_df = ticker_df.select(
    F.concat_ws("_", "company_name", F.date_format("Date", "yyyy-MM-dd")).alias("id"),
    F.lit("TradingDay").alias("type"),
    F.col("company_name").alias("ticker"),
    F.col("Date").alias("date"),
    "price_open", "price_close", "volume",
    F.round((F.col("price_close") - F.col("price_open")) / F.col("price_open") * 100, 4).alias("daily_return")
)

# ── TRADED_ON Edges ──
# Links each Company to its TradingDay nodes. Primary ownership relationship enabling "show me NVDA's
# trading history" queries.
traded_on_edges = ticker_df.select(
    F.col("company_name").alias("src"),
    F.concat_ws("_", "company_name", F.date_format("Date", "yyyy-MM-dd")).alias("dst"),
    F.lit("TRADED_ON").alias("relationship")
)

# ── NEXT_DAY Edges ──
# Temporal chain: each TradingDay points to the next calendar trading day for the SAME company. This enables
# streak/momentum queries like "longest consecutive days of positive returns" via graph traversal instead of
# window functions. We use lead() over a per-company date-ordered window to find the next date, then filter
# out the last day per company (where next_date is null).
day_window = Window.partitionBy("company_name").orderBy("Date")
with_next = ticker_df.withColumn("next_date", F.lead("Date").over(day_window))
next_day_edges = with_next.filter(F.col("next_date").isNotNull()).select(
    F.concat_ws("_", "company_name", F.date_format("Date", "yyyy-MM-dd")).alias("src"),
    F.concat_ws("_", "company_name", F.date_format("next_date", "yyyy-MM-dd")).alias("dst"),
    F.lit("NEXT_DAY").alias("relationship")
)

# Expect: 7 Company vertices, ~1750 TradingDay vertices, ~1750 TRADED_ON edges, ~1743 NEXT_DAY edges
# (7 fewer = one chain-end per company)
print(f"\nGraph components:")
print(f"  Company vertices: {companies_df.count()}")
print(f"  TradingDay vertices: {trading_days_df.count()}")
print(f"  TRADED_ON edges: {traded_on_edges.count()}")
print(f"  NEXT_DAY edges: {next_day_edges.count()}")

In [0]:
# --- Persist Graph as Delta Tables ---
# We flatten all vertex types into a single table with a union-compatible schema. Properties that don't
# apply to a given vertex type are stored as NULL. All property columns are cast to STRING for uniform
# storage. A `name` column labels the hub nodes added in the enrichment cell below.
#
# This "wide + sparse" layout is simpler than separate tables per vertex type and works well because the
# pro app issues targeted SQL with WHERE type = '...'.

vertices_table = f"{CATALOG}.{SCHEMA}.graphrag_vertices"
edges_table = f"{CATALOG}.{SCHEMA}.graphrag_edges"

# ── Vertices: union Company + TradingDay nodes ──
# Company nodes carry fundamentals (market_cap, beta, pe_*) but no date/price props. TradingDay nodes carry
# date/price/volume/return but no fundamental props. `name` is NULL for both (used only by hub nodes).
all_vertices = companies_df.select("id", "type", "ticker",
    F.col("market_cap").cast("string").alias("prop_market_cap"),
    F.col("beta").cast("string").alias("prop_beta"),
    F.col("pe_trailing").cast("string").alias("prop_pe_trailing"),
    F.col("pe_forward").cast("string").alias("prop_pe_forward"),
    F.col("ev_ebitda").cast("string").alias("prop_ev_ebitda"),
    F.lit(None).cast("string").alias("prop_date"),
    F.lit(None).cast("string").alias("prop_price_close"),
    F.lit(None).cast("string").alias("prop_volume"),
    F.lit(None).cast("string").alias("prop_daily_return"),
    F.lit(None).cast("string").alias("name")
).unionByName(
    trading_days_df.select("id", "type", "ticker",
        F.lit(None).cast("string").alias("prop_market_cap"),
        F.lit(None).cast("string").alias("prop_beta"),
        F.lit(None).cast("string").alias("prop_pe_trailing"),
        F.lit(None).cast("string").alias("prop_pe_forward"),
        F.lit(None).cast("string").alias("prop_ev_ebitda"),
        F.col("date").cast("string").alias("prop_date"),
        F.col("price_close").cast("string").alias("prop_price_close"),
        F.col("volume").cast("string").alias("prop_volume"),
        F.col("daily_return").cast("string").alias("prop_daily_return"),
        F.lit(None).cast("string").alias("name")
    )
)

# ── Edges: union TRADED_ON + NEXT_DAY relationships ──
# Both edge types share the same (src, dst, relationship) schema. `weight` is NULL here; the enrichment
# cell uses it to carry correlation strength.
all_edges = traded_on_edges.unionByName(next_day_edges).withColumn(
    "weight", F.lit(None).cast("double")
)

# Write with overwrite mode so the notebook is fully idempotent. Delta gives us ACID guarantees + time-travel.
all_vertices.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(vertices_table)
all_edges.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(edges_table)

print(f"✅ Wrote {vertices_table} ({all_vertices.count()} rows)")
print(f"✅ Wrote {edges_table} ({all_edges.count()} rows)")

In [ ]:
# --- Enrich the graph: sectors, risk/valuation tiers, MAG7 index, correlation web ---
# Beyond the Company→TradingDay star, we add company-level structure so the knowledge graph has multiple
# "centers of gravity" and cross-company relationships the pro app visualizes:
#   • Index  : a single MAG7 super-node every company is a MEMBER_OF
#   • Sector : 3 sector hubs companies BELONG_TO (static GICS-style mapping)
#   • BetaTier / ValuationTier : risk + valuation buckets (computed from beta / forward P/E)
#   • CORRELATED_WITH : weighted edges between companies whose daily returns correlate >= 0.6
# These rows are *appended* to the tables written above (which overwrite), so re-running cells 4+5 stays
# idempotent. The pro app's /api/graph surfaces them as hubs + a weighted correlation web.

PROP_COLS = ["prop_market_cap", "prop_beta", "prop_pe_trailing", "prop_pe_forward",
             "prop_ev_ebitda", "prop_date", "prop_price_close", "prop_volume", "prop_daily_return"]

# ── Hub vertices (index / sectors / tiers) ──
hub_specs = [
    ("IDX::MAG7", "Index", "MAG7"),
    ("SEC::Technology", "Sector", "Technology"),
    ("SEC::Communication Services", "Sector", "Communication Services"),
    ("SEC::Consumer Cyclical", "Sector", "Consumer Cyclical"),
    ("BETA::Low", "BetaTier", "Low Beta"),
    ("BETA::Moderate", "BetaTier", "Moderate Beta"),
    ("BETA::High", "BetaTier", "High Beta"),
    ("VAL::Value", "ValuationTier", "Value"),
    ("VAL::Core", "ValuationTier", "Core"),
    ("VAL::Premium", "ValuationTier", "Premium"),
]
hub_vertices = spark.createDataFrame(hub_specs, ["id", "type", "name"]).withColumn(
    "ticker", F.lit(None).cast("string")
)
for c in PROP_COLS:
    hub_vertices = hub_vertices.withColumn(c, F.lit(None).cast("string"))
hub_vertices = hub_vertices.select("id", "type", "ticker", *PROP_COLS, "name")
hub_vertices.write.format("delta").mode("append").saveAsTable(vertices_table)

# ── Sector membership (static, GICS-style mapping for the Mag-7) ──
SECTOR_MAP = {
    "AAPL": "SEC::Technology", "MSFT": "SEC::Technology", "NVDA": "SEC::Technology",
    "GOOGL": "SEC::Communication Services", "META": "SEC::Communication Services",
    "AMZN": "SEC::Consumer Cyclical", "TSLA": "SEC::Consumer Cyclical",
}
belongs_to = spark.createDataFrame(list(SECTOR_MAP.items()), ["src", "dst"]) \
    .withColumn("relationship", F.lit("BELONGS_TO")).withColumn("weight", F.lit(None).cast("double"))

# ── MAG7 index membership ──
member_of = companies_df.select(F.col("ticker").alias("src")) \
    .withColumn("dst", F.lit("IDX::MAG7")).withColumn("relationship", F.lit("MEMBER_OF")) \
    .withColumn("weight", F.lit(None).cast("double"))

# ── Risk (beta) + valuation (forward P/E) tiers, computed from the fundamentals ──
has_beta = companies_df.select(
    F.col("ticker").alias("src"),
    F.when(F.col("beta") < 1.15, "BETA::Low")
     .when(F.col("beta") <= 1.5, "BETA::Moderate")
     .otherwise("BETA::High").alias("dst"),
).withColumn("relationship", F.lit("HAS_BETA")).withColumn("weight", F.lit(None).cast("double"))

has_valuation = companies_df.select(
    F.col("ticker").alias("src"),
    F.when(F.col("pe_forward") < 30, "VAL::Value")
     .when(F.col("pe_forward") <= 40, "VAL::Core")
     .otherwise("VAL::Premium").alias("dst"),
).withColumn("relationship", F.lit("HAS_VALUATION")).withColumn("weight", F.lit(None).cast("double"))

# ── Correlation web: pairwise daily-return correlation, kept when >= 0.6, weighted ──
a = trading_days_df.select("ticker", "date", "daily_return").alias("a")
b = trading_days_df.select("ticker", "date", "daily_return").alias("b")
pairs = a.join(b, (F.col("a.date") == F.col("b.date")) & (F.col("a.ticker") < F.col("b.ticker")))
correlated = pairs.groupBy(F.col("a.ticker").alias("src"), F.col("b.ticker").alias("dst")) \
    .agg(F.round(F.corr("a.daily_return", "b.daily_return"), 3).alias("weight")) \
    .where(F.col("weight") >= 0.6) \
    .withColumn("relationship", F.lit("CORRELATED_WITH")) \
    .select("src", "dst", "relationship", "weight")

enrichment_edges = member_of.unionByName(belongs_to).unionByName(has_beta) \
    .unionByName(has_valuation).unionByName(correlated)
enrichment_edges.write.format("delta").mode("append").saveAsTable(edges_table)

print(f"✅ Appended {hub_vertices.count()} hub vertices (index / sectors / tiers)")
print(f"✅ Appended {enrichment_edges.count()} structural + correlation edges")
print(f"   Correlation edges: {correlated.count()} (return corr >= 0.6)")

In [ ]:
# --- Create GraphRAG SQL UC functions (replaces the FastMCP server approach) ---
# The Supervisor consumes the knowledge graph through SQL Unity Catalog functions over
# graphrag_vertices, NOT an external MCP server. External MCP on Databricks Apps is
# unusable through the Agent Bricks MCP proxy on streaming requests (httpx
# ResponseNotRead; see ML-63338). UC function tools register reliably. Notebook 04 adds
# these as Supervisor tools; notebook 06 grants the app service principal EXECUTE.
VERTICES = f"`{CATALOG}`.`{SCHEMA}`.graphrag_vertices"

spark.sql(f"""
CREATE OR REPLACE FUNCTION `{CATALOG}`.`{SCHEMA}`.get_company_summary(p_ticker STRING)
RETURNS STRING
COMMENT 'Stock knowledge-graph summary for one company: fundamentals (market cap, beta, trailing/forward P/E, EV/EBITDA) plus its 10 most recent trading days (date | close | volume). Input: a single ticker such as NVDA, AAPL, MSFT, GOOGL, META, AMZN, TSLA.'
RETURN (
  SELECT CONCAT_WS(char(10),
    CONCAT('=== ', UPPER(p_ticker), ' (knowledge graph) ==='),
    CONCAT('market_cap=', MAX(CASE WHEN type='Company' THEN prop_market_cap END),
           ' beta=', MAX(CASE WHEN type='Company' THEN prop_beta END),
           ' pe_trailing=', MAX(CASE WHEN type='Company' THEN prop_pe_trailing END),
           ' pe_forward=', MAX(CASE WHEN type='Company' THEN prop_pe_forward END),
           ' ev_ebitda=', MAX(CASE WHEN type='Company' THEN prop_ev_ebitda END)),
    'recent_days (date | close | volume):',
    ARRAY_JOIN(SLICE(ARRAY_SORT(
        COLLECT_LIST(CASE WHEN type='TradingDay' THEN CONCAT_WS(' | ', prop_date, prop_price_close, prop_volume) END),
        (l, r) -> CASE WHEN l > r THEN -1 WHEN l < r THEN 1 ELSE 0 END), 1, 10), char(10))
  )
  FROM {VERTICES}
  WHERE UPPER(ticker) = UPPER(p_ticker)
)
""")

spark.sql(f"""
CREATE OR REPLACE FUNCTION `{CATALOG}`.`{SCHEMA}`.compare_companies(p_tickers STRING)
RETURNS STRING
COMMENT 'Compare multiple companies from the stock knowledge graph by fundamentals (market cap, beta, trailing/forward P/E, EV/EBITDA), sorted by market cap. Input: comma-separated tickers, e.g. "NVDA,AAPL,MSFT".'
RETURN (
  SELECT CONCAT_WS(char(10),
    '=== Company comparison (knowledge graph) ===',
    'ticker | market_cap | beta | pe_trailing | pe_forward | ev_ebitda',
    ARRAY_JOIN(TRANSFORM(ARRAY_SORT(
        COLLECT_LIST(NAMED_STRUCT(
          'mc', CAST(prop_market_cap AS DOUBLE),
          'line', CONCAT_WS(' | ', ticker, prop_market_cap, prop_beta, prop_pe_trailing, prop_pe_forward, prop_ev_ebitda))),
        (l, r) -> CASE WHEN l.mc > r.mc THEN -1 WHEN l.mc < r.mc THEN 1 ELSE 0 END),
      x -> x.line), char(10))
  )
  FROM {VERTICES}
  WHERE type='Company'
    AND ARRAY_CONTAINS(SPLIT(UPPER(REPLACE(p_tickers, ' ', '')), ','), UPPER(ticker))
)
""")

print("Created UC functions: get_company_summary, compare_companies")
print(spark.sql(f"SELECT `{CATALOG}`.`{SCHEMA}`.get_company_summary('NVDA')").collect()[0][0][:300])


In [0]:
# --- Smoke Tests ---
# Validate the graph tables directly via Spark SQL before relying on the MCP server. These queries mirror
# what the app's three tools execute, ensuring data is correct at the source. If these fail, the MCP tools
# will also fail.

print("Testing graph queries...")

print("\nTest 1: Company summary (NVDA)")  # validates get_company_summary() data path
result = spark.sql(f"""
    SELECT ticker, prop_market_cap as market_cap, prop_beta as beta,
           prop_pe_trailing as pe_trailing, prop_pe_forward as pe_forward
    FROM {CATALOG}.{SCHEMA}.graphrag_vertices
    WHERE type = 'Company' AND ticker = 'NVDA'
""").show(truncate=False)

print("\nTest 2: Recent NVDA trading days")  # validates the recent-trading portion of tools
spark.sql(f"""
    SELECT prop_date as date, prop_price_close as close, 
           prop_volume as volume, prop_daily_return as return_pct
    FROM {CATALOG}.{SCHEMA}.graphrag_vertices
    WHERE type = 'TradingDay' AND ticker = 'NVDA'
    ORDER BY prop_date DESC
    LIMIT 5
""").show(truncate=False)

# Each company should have ~250 TRADED_ON edges (one per trading day). Significant deviation = data issue.
print("\nTest 3: TRADED_ON edges per company")
spark.sql(f"""
    SELECT src as company, COUNT(*) as trading_days
    FROM {CATALOG}.{SCHEMA}.graphrag_edges
    WHERE relationship = 'TRADED_ON'
    GROUP BY src
    ORDER BY src
""").show()

print("✅ All tests passed")

In [0]:
%pip install networkx --quiet
dbutils.library.restartPython()

In [0]:
# --- Network Graph Visualization ---
# Renders the property graph as a network diagram using networkx + matplotlib.
# Shows all 7 Company hub nodes plus the N most recent TradingDay nodes per company,
# with TRADED_ON (solid) and NEXT_DAY (dashed) edges.

import sys, math
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Re-load config after Python restart
sys.path.append('..')
exec(open('../config.py').read())
CATALOG = catalog
SCHEMA = schema

# ── Parameters ──
N_DAYS = 5  # recent trading days per company to display

# ── Load graph tables ──
vertices_df = spark.table(f"{CATALOG}.{SCHEMA}.graphrag_vertices")
edges_df = spark.table(f"{CATALOG}.{SCHEMA}.graphrag_edges")

# ── Select visible nodes ──
company_pd = vertices_df.filter("type = 'Company'").select("id", "ticker").toPandas()

w_rank = Window.partitionBy("ticker").orderBy(F.col("prop_date").desc())
recent_pd = (
    vertices_df
    .filter("type = 'TradingDay'")
    .withColumn("rn", F.row_number().over(w_rank))
    .filter(f"rn <= {N_DAYS}")
    .select("id", "ticker", "prop_date", "prop_daily_return")
    .toPandas()
)

visible_ids = set(company_pd["id"].tolist() + recent_pd["id"].tolist())

# ── Filter edges to visible subgraph ──
edges_pd = edges_df.toPandas()
vis_edges = edges_pd[edges_pd["src"].isin(visible_ids) & edges_pd["dst"].isin(visible_ids)]

# ── Build networkx DiGraph ──
G = nx.DiGraph()

for _, r in company_pd.iterrows():
    G.add_node(r["id"], node_type="Company", ticker=r["ticker"], label=r["ticker"])

for _, r in recent_pd.iterrows():
    date_short = str(r["prop_date"])[:10] if r["prop_date"] else ""
    G.add_node(r["id"], node_type="TradingDay", ticker=r["ticker"], label=date_short)

for _, r in vis_edges.iterrows():
    G.add_edge(r["src"], r["dst"], relationship=r["relationship"])

# ── Colour palette per ticker ──
palette = {
    "AAPL": "#A2AAAD", "MSFT": "#00A4EF", "NVDA": "#76B900",
    "GOOGL": "#4285F4", "META": "#0668E1", "AMZN": "#FF9900", "TSLA": "#CC0000"
}

# ── Layout: place Company nodes in a circle, spring-push TradingDays outward ──
company_ids = [n for n, d in G.nodes(data=True) if d["node_type"] == "Company"]
td_ids      = [n for n, d in G.nodes(data=True) if d["node_type"] == "TradingDay"]

ticker_to_cid = {G.nodes[n]["ticker"]: n for n in company_ids}

fixed_pos = {}
for i, cid in enumerate(sorted(company_ids)):
    angle = 2 * math.pi * i / len(company_ids)
    fixed_pos[cid] = (3 * math.cos(angle), 3 * math.sin(angle))

initial_pos = {**fixed_pos}
for n in td_ids:
    ticker = G.nodes[n]["ticker"]
    parent_cid = ticker_to_cid.get(ticker)
    if parent_cid and parent_cid in fixed_pos:
        cx, cy = fixed_pos[parent_cid]
    else:
        cx, cy = 0, 0
    initial_pos[n] = (cx + 0.3 * (hash(n) % 10 - 5), cy + 0.3 * (hash(n) % 7 - 3))

pos = nx.spring_layout(G, pos=initial_pos, fixed=company_ids, k=1.8, iterations=80, seed=42)

# ── Draw ──
fig, ax = plt.subplots(figsize=(18, 14))

traded_on = [(u, v) for u, v, d in G.edges(data=True) if d["relationship"] == "TRADED_ON"]
next_day  = [(u, v) for u, v, d in G.edges(data=True) if d["relationship"] == "NEXT_DAY"]

nx.draw_networkx_edges(G, pos, edgelist=traded_on, alpha=0.25, edge_color="#888888",
                       style="solid", arrows=True, arrowsize=12, ax=ax,
                       connectionstyle="arc3,rad=0.05")
nx.draw_networkx_edges(G, pos, edgelist=next_day, alpha=0.6, edge_color="#E74C3C",
                       style="dashed", arrows=True, arrowsize=12, width=1.5, ax=ax)

nx.draw_networkx_nodes(G, pos, nodelist=company_ids, node_size=1400,
                       node_color=[palette.get(G.nodes[n]["ticker"], "#999") for n in company_ids],
                       edgecolors="black", linewidths=2.5, ax=ax)

nx.draw_networkx_nodes(G, pos, nodelist=td_ids, node_size=350,
                       node_color=[palette.get(G.nodes[n]["ticker"], "#999") for n in td_ids],
                       alpha=0.75, edgecolors="gray", linewidths=0.5, ax=ax)

nx.draw_networkx_labels(G, pos,
                        labels={n: G.nodes[n]["label"] for n in company_ids},
                        font_size=13, font_weight="bold", ax=ax)
nx.draw_networkx_labels(G, pos,
                        labels={n: G.nodes[n]["label"] for n in td_ids},
                        font_size=7, font_color="#333333", ax=ax)

# Legend
handles = [mpatches.Patch(color=c, label=t) for t, c in palette.items()]
handles += [
    plt.Line2D([0], [0], color="#888888", lw=1.5, label="TRADED_ON"),
    plt.Line2D([0], [0], color="#E74C3C", lw=1.5, ls="--", label="NEXT_DAY (temporal chain)"),
]
ax.legend(handles=handles, loc="upper left", fontsize=9, framealpha=0.9, title="Legend")

total_v = vertices_df.count()
total_e = edges_df.count()

ax.set_title(
    f"Mag-7 Stock Knowledge Graph \u2014 {len(company_ids)} Companies \u00d7 {N_DAYS} Recent Trading Days",
    fontsize=16, fontweight="bold", pad=20
)
ax.text(0.5, -0.02,
        f"Full graph: {total_v:,} vertices \u00b7 {total_e:,} edges  |  "
        f"Showing {G.number_of_nodes()} nodes \u00b7 {G.number_of_edges()} edges",
        transform=ax.transAxes, ha="center", fontsize=10, color="gray")

ax.axis("off")
plt.tight_layout()
plt.show()

## Setup Complete

### Artifacts Created

| Layer | Resource | Details |
|-------|----------|---------|
| **Delta Tables** | `{catalog}.{schema}.graphrag_vertices` | 1,757 rows — 7 Company + 1,750 TradingDay nodes with properties stored as STRING columns |
| | `{catalog}.{schema}.graphrag_edges` | 3,493 rows — 1,750 TRADED_ON + 1,743 NEXT_DAY relationships |
| **Databricks App** | `{GRAPHRAG_APP_NAME}` | FastMCP server running on SSE transport with 3 tools: `query_graph`, `get_company_summary`, `compare_companies` |
| **UC Connection** | `ontobricks-graphrag` | HTTP connection with bearer token auth, pointing to the app's `/mcp/sse` endpoint |

### How It Connects to the Supervisor

Notebook **`04_instructor_setup_sa`** discovers UC connections and registers them as child agents. When it finds `ontobricks-graphrag`, it adds an agent entry like:

```python
{
    "name": "GraphRAG_Stock_Knowledge",
    "agent_type": "external-mcp-server",
    "external_mcp_server": {"connection_name": "ontobricks-graphrag"}
}
```

The Supervisor then routes graph-related questions to this MCP agent.

### Example Queries via the Supervisor Agent

| Query | Routed To | What Happens |
|-------|-----------|---------------|
| "Give me NVDA's fundamentals and recent prices" | `get_company_summary("NVDA")` | Returns Company node props + last 10 TradingDay nodes |
| "Compare AAPL, MSFT, and NVDA" | `compare_companies("AAPL,MSFT,NVDA")` | Queries Company vertices, returns side-by-side table |
| "What does the knowledge graph contain?" | `query_graph("...")` | Returns schema description with node/edge counts |
| "Which stock has the highest P/E ratio?" | `query_graph("...")` | Keyword routing → company comparison sorted by metric |

### Cleanup

To remove all artifacts created by this notebook:

```sql
-- Drop graph tables
DROP TABLE IF EXISTS {catalog}.{schema}.graphrag_vertices;
DROP TABLE IF EXISTS {catalog}.{schema}.graphrag_edges;

-- Drop UC connection (run from a Python cell)
-- w.api_client.do("DELETE", f"/api/2.1/unity-catalog/connections/ontobricks-graphrag")

-- Delete the Databricks App (run from a Python cell)
-- w.api_client.do("DELETE", f"/api/2.0/apps/{GRAPHRAG_APP_NAME}")
```

### Related Notebooks

* **`00_tables_setup`** — Creates the source `ticker_data` table (prerequisite)
* **`04_instructor_setup_sa`** — Creates the Supervisor Agent and wires in MCP connections
* **`05_create_mcp_server_OPTIONAL`** — Adds the You.com web search MCP server (sibling pattern)
* **`05b_create_mcp_neo4j_graph_OPTIONAL`** — Neo4j-based alternative to this notebook (requires external AuraDB instance)